# Snowflake Feature Store Quickstart

| Private Preview Feature | Description |
|---|---|
| **Low-latency feature serving** | Online feature serving backed by a managed Postgres instance (<20 ms p50) |
| **Time-windowed aggregation** | Rolling-window aggregate features (SUM, COUNT, AVG) with configurable tile sizes |
| **Stream feature views** | Stream event ingestion via StreamSource with 2-3 second end-to-end freshness |
| **REST ingest & query API** | HTTP endpoints for streaming data ingestion and online feature retrieval |

This notebook demonstrates these capabilities using an **e-commerce clickstream** dataset.
You'll learn how to:

1. **Set up** a Feature Store with an Online Service (Postgres-backed)
2. **Create batch feature views** — passthrough columns from purchase data
3. **Create tiled feature views** — time-windowed aggregations (SUM, COUNT, AVG)
4. **Create stream feature views** — stream event ingestion via REST API
5. **Read online features** — low-latency lookups for inference
6. **Generate training sets** — point-in-time correct joins for ML

**Use case**: An e-commerce platform tracks user browsing events and purchase orders.
We build features for a **session conversion prediction** model that scores whether
a user's browsing session will result in a purchase.

---

> **Private Preview** — This notebook uses features that are currently in private preview
> and available to selected accounts only. They require `snowflake-ml-python >= 1.36`.
> For full details, see the
> [Online Feature Store with Snowflake Postgres](https://docs.snowflake.com/en/LIMITEDACCESS/online-feature-store-private-preview)
> documentation.


In [ ]:
!pip install --upgrade --force-reinstall "snowflake-ml-python>=1.36"

### Configure Snowflake PAT

Online feature serving authenticates via a
[Programmatic Access Token (PAT)](https://docs.snowflake.com/en/user-guide/programmatic-access-tokens).
Set it as the `SNOWFLAKE_PAT` environment variable before connecting.

> **Network policy:** Make sure you are running this notebook from a machine whose
> IP address is allowed by your account's network policy. PAT-based authentication
> will fail if your IP is not on the allowlist. See
> [Network policy requirements](https://docs.snowflake.com/en/user-guide/programmatic-access-tokens#network-policy-requirements)
> for details.

If you already have the variable set in your shell, you can skip this cell.

In [ ]:
import os

if not os.environ.get('SNOWFLAKE_PAT'):
    os.environ['SNOWFLAKE_PAT'] = '<ENTER_YOUR_SNOWFLAKE_PAT>'

print('SNOWFLAKE_PAT configured')

SNOWFLAKE_PAT configured


## 1. Environment Setup

Before using Feature Store private preview features, ensure the following:

- Your account has been **enabled** for the Feature Store private preview.
- `snowflake-ml-python >= 1.36` is installed.
- A **Programmatic Access Token (PAT)** is set as `SNOWFLAKE_PAT` for online
  feature serving authentication.

We also create **producer** and **consumer** roles to control access to the online
service, and grant them to a parent role (e.g. a team or project role — avoid
granting directly to built-in roles like `SYSADMIN` or `ACCOUNTADMIN` in production).
The equivalent SQL is:

```sql
CREATE ROLE IF NOT EXISTS DEMO_PRODUCER_ROLE;
CREATE ROLE IF NOT EXISTS DEMO_CONSUMER_ROLE;
GRANT ROLE DEMO_PRODUCER_ROLE TO ROLE PROD_ROLE;
GRANT ROLE DEMO_CONSUMER_ROLE TO ROLE PROD_ROLE;
```

In [ ]:
import os
import time
import json
from datetime import datetime, timedelta

from snowflake.snowpark import Session
from snowflake.snowpark.functions import col
from snowflake.ml.feature_store import (
    FeatureStore,
    FeatureView,
    Entity,
    Feature,
    CreationMode,
    OnlineConfig,
    OnlineStoreType,
    StreamSource,
    StreamConfig,
    online_service,
)
from snowflake.snowpark.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, TimestampTimeZone,
)

def day(offset):
    """Return date string for today + offset days."""
    return (datetime.now() + timedelta(days=offset)).strftime('%Y-%m-%d')

# Configuration
DATABASE = 'CLICKSTREAM_FS_DEMO'
SCHEMA = 'FEATURE_STORE'
SOURCE_SCHEMA = 'CLICKSTREAM_DATA'
WAREHOUSE = 'DEMO_WH'
PRODUCER_ROLE = 'DEMO_PRODUCER_ROLE'
CONSUMER_ROLE = 'DEMO_CONSUMER_ROLE'

PARENT_ROLE = 'PROD_ROLE'

In [ ]:
# Connect and create infrastructure
from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql(f'CREATE DATABASE IF NOT EXISTS {DATABASE}').collect()
session.sql(f'CREATE SCHEMA IF NOT EXISTS {DATABASE}.{SOURCE_SCHEMA}').collect()
session.sql(f'CREATE SCHEMA IF NOT EXISTS {DATABASE}.{SCHEMA}').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE SCHEMA {DATABASE}.{SCHEMA}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()

# Create roles and grant to the parent role
for role in [PRODUCER_ROLE, CONSUMER_ROLE]:
    session.sql(f'CREATE ROLE IF NOT EXISTS {role}').collect()
    session.sql(f'GRANT ROLE {role} TO ROLE {PARENT_ROLE}').collect()

print(f'Connected | account={session.get_current_account()}, role={session.get_current_role()}')

Connected | account="feature_store_vnext", role="ACCOUNTADMIN"


### Initialize the Feature Store

The `FeatureStore` object is the central entry point for all operations — registering
entities, creating feature views, and reading features. It manages a dedicated schema
within a Snowflake database.

Use `CreationMode.CREATE_IF_NOT_EXIST` to create the schema and internal metadata
tables on first run, or connect to an existing Feature Store on subsequent runs.

In [5]:
# Initialize Feature Store
fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print('Feature Store initialized')

Feature Store initialized


### Create the Online Service

The **online service** is a managed Postgres-backed serving layer that provides
low-latency feature reads (<20 ms p50). It must be created **once per Feature Store**
before registering any feature views with online serving enabled.

The service provisions two REST endpoints:
- **Ingest** — accepts streaming events to be ingested for feature updates
- **Query** — low-latency serving of online features

First creation takes several minutes. We poll `get_online_service_status()` until
the service reaches `RUNNING`.

In [6]:
# Create Online Service (Postgres-backed)
# This provisions the ingest + query REST endpoints for low-latency online reads.
try:
    create_result = fs.create_online_service(PRODUCER_ROLE, CONSUMER_ROLE)
    print(f'Create result: {create_result}')
except Exception as e:
    if 'already exists' in str(e):
        print('Online Service already exists — skipping creation.')
    else:
        raise

# Wait for Online Service to reach RUNNING state
for i in range(30):
    status = fs.get_online_service_status()
    ep_names = [ep.name for ep in status.endpoints]
    print(f'  [{i}] Status: {status.status} | Endpoints: {ep_names}')
    if status.status == 'RUNNING' and online_service.endpoint_url(status, 'query'):
        query_url = online_service.endpoint_url(status, 'query')
        ingest_url = online_service.endpoint_url(status, 'ingest')
        print(f'Online Service RUNNING')
        print(f'  Query:  {query_url}')
        print(f'  Ingest: {ingest_url}')
        break
    time.sleep(30)
else:
    print('WARNING: Online Service did not reach RUNNING in time. Online features may not work.')

Online Service already exists — skipping creation.
  [0] Status: RUNNING | Endpoints: ['ingest', 'query']
Online Service RUNNING
  Query:  https://fxhi5jn-sfengineering-feature-store-vnext.awsuswest2qa6.test-snowflakecomputing.app
  Ingest: https://bxhi5jn-sfengineering-feature-store-vnext.awsuswest2qa6.test-snowflakecomputing.app


---
## 2. Create Clickstream Source Data

We create three source tables that simulate an e-commerce platform:

| Table | Description | Key Columns |
|-------|-------------|-------------|
| **ORDERS** | Purchase transactions | USER_ID, ORDER_TS, TOTAL_AMT |
| **EVENTS** | Clickstream browsing events | USER_ID, EVENT_TS, EVENT_TYPE, SESSION_ID |
| **SESSIONS** | Browsing sessions with conversion labels | USER_ID, SESSION_START_TS, IS_CONVERTED |

In [7]:
# === ORDERS: Purchase transactions ===
ORDERS_TABLE = f'{DATABASE}.{SOURCE_SCHEMA}.ORDERS'

session.sql(f'''
CREATE OR REPLACE TABLE {ORDERS_TABLE} (
    ORDER_ID    VARCHAR,
    USER_ID     VARCHAR,
    ORDER_TS    TIMESTAMP_NTZ,
    TOTAL_AMT   FLOAT,
    STATUS      VARCHAR
)
''').collect()

session.sql(f'''
INSERT INTO {ORDERS_TABLE} VALUES
    ('ord_001', 'user_1', '{day(-14)} 09:00:00', 45.00,  'completed'),
    ('ord_002', 'user_1', '{day(-10)} 14:00:00', 120.50, 'completed'),
    ('ord_003', 'user_1', '{day(-7)}  11:30:00', 35.00,  'completed'),
    ('ord_004', 'user_1', '{day(-3)}  16:00:00', 89.99,  'completed'),
    ('ord_005', 'user_1', '{day(-1)}  10:00:00', 210.00, 'completed'),
    ('ord_006', 'user_2', '{day(-12)} 08:00:00', 550.00, 'completed'),
    ('ord_007', 'user_2', '{day(-8)}  19:00:00', 75.00,  'completed'),
    ('ord_008', 'user_2', '{day(-5)}  12:00:00', 320.00, 'completed'),
    ('ord_009', 'user_2', '{day(-2)}  15:30:00', 45.50,  'completed'),
    ('ord_010', 'user_3', '{day(-20)} 10:00:00', 1200.00,'completed'),
    ('ord_011', 'user_3', '{day(-6)}  09:00:00', 85.00,  'completed'),
    ('ord_012', 'user_3', '{day(-1)}  14:00:00', 150.00, 'completed'),
    ('ord_013', 'user_4', '{day(-9)}  11:00:00', 60.00,  'completed'),
    ('ord_014', 'user_4', '{day(-4)}  17:00:00', 95.00,  'completed'),
    ('ord_015', 'user_5', '{day(-15)} 13:00:00', 400.00, 'completed'),
    ('ord_016', 'user_5', '{day(-2)}  09:30:00', 175.00, 'completed')
''').collect()

print(f'ORDERS: {session.table(ORDERS_TABLE).count()} rows')
session.table(ORDERS_TABLE).show()

ORDERS: 16 rows
--------------------------------------------------------------------------
|"ORDER_ID"  |"USER_ID"  |"ORDER_TS"           |"TOTAL_AMT"  |"STATUS"   |
--------------------------------------------------------------------------
|ord_001     |user_1     |2026-04-06 09:00:00  |45.0         |completed  |
|ord_002     |user_1     |2026-04-10 14:00:00  |120.5        |completed  |
|ord_003     |user_1     |2026-04-13 11:30:00  |35.0         |completed  |
|ord_004     |user_1     |2026-04-17 16:00:00  |89.99        |completed  |
|ord_005     |user_1     |2026-04-19 10:00:00  |210.0        |completed  |
|ord_006     |user_2     |2026-04-08 08:00:00  |550.0        |completed  |
|ord_007     |user_2     |2026-04-12 19:00:00  |75.0         |completed  |
|ord_008     |user_2     |2026-04-15 12:00:00  |320.0        |completed  |
|ord_009     |user_2     |2026-04-18 15:30:00  |45.5         |completed  |
|ord_010     |user_3     |2026-03-31 10:00:00  |1200.0       |completed  |
---------

In [8]:
# === EVENTS: Clickstream browsing events ===
EVENTS_TABLE = f'{DATABASE}.{SOURCE_SCHEMA}.EVENTS'

session.sql(f'''
CREATE OR REPLACE TABLE {EVENTS_TABLE} (
    EVENT_ID    VARCHAR,
    USER_ID     VARCHAR,
    SESSION_ID  VARCHAR,
    EVENT_TS    TIMESTAMP_NTZ,
    EVENT_TYPE  VARCHAR,
    PRODUCT_ID  VARCHAR
)
''').collect()

session.sql(f'''
INSERT INTO {EVENTS_TABLE} VALUES
    ('evt_001', 'user_1', 'sess_01', '{day(-3)} 15:30:00', 'page_view',   'prod_A'),
    ('evt_002', 'user_1', 'sess_01', '{day(-3)} 15:31:00', 'page_view',   'prod_B'),
    ('evt_003', 'user_1', 'sess_01', '{day(-3)} 15:35:00', 'add_to_cart', 'prod_A'),
    ('evt_004', 'user_1', 'sess_01', '{day(-3)} 15:40:00', 'purchase',    'prod_A'),
    ('evt_005', 'user_1', 'sess_02', '{day(-1)} 09:50:00', 'page_view',   'prod_C'),
    ('evt_006', 'user_1', 'sess_02', '{day(-1)} 09:55:00', 'page_view',   'prod_D'),
    ('evt_007', 'user_1', 'sess_02', '{day(-1)} 10:00:00', 'purchase',    'prod_C'),
    ('evt_008', 'user_2', 'sess_03', '{day(-5)} 11:45:00', 'page_view',   'prod_B'),
    ('evt_009', 'user_2', 'sess_03', '{day(-5)} 11:50:00', 'add_to_cart', 'prod_B'),
    ('evt_010', 'user_2', 'sess_03', '{day(-5)} 12:00:00', 'purchase',    'prod_B'),
    ('evt_011', 'user_2', 'sess_04', '{day(-2)} 15:00:00', 'page_view',   'prod_E'),
    ('evt_012', 'user_2', 'sess_04', '{day(-2)} 15:10:00', 'page_view',   'prod_A'),
    ('evt_013', 'user_2', 'sess_04', '{day(-2)} 15:30:00', 'purchase',    'prod_E'),
    ('evt_014', 'user_3', 'sess_05', '{day(-6)} 08:30:00', 'page_view',   'prod_D'),
    ('evt_015', 'user_3', 'sess_05', '{day(-6)} 08:45:00', 'page_view',   'prod_C'),
    ('evt_016', 'user_3', 'sess_05', '{day(-6)} 09:00:00', 'purchase',    'prod_D'),
    ('evt_017', 'user_3', 'sess_06', '{day(-1)} 13:30:00', 'page_view',   'prod_A'),
    ('evt_018', 'user_3', 'sess_06', '{day(-1)} 13:45:00', 'add_to_cart', 'prod_A'),
    ('evt_019', 'user_3', 'sess_06', '{day(-1)} 14:00:00', 'purchase',    'prod_A'),
    ('evt_020', 'user_4', 'sess_07', '{day(-4)} 16:30:00', 'page_view',   'prod_B'),
    ('evt_021', 'user_4', 'sess_07', '{day(-4)} 16:45:00', 'page_view',   'prod_C'),
    ('evt_022', 'user_4', 'sess_07', '{day(-4)} 17:00:00', 'purchase',    'prod_C'),
    ('evt_023', 'user_5', 'sess_08', '{day(-2)} 09:00:00', 'page_view',   'prod_A'),
    ('evt_024', 'user_5', 'sess_08', '{day(-2)} 09:15:00', 'page_view',   'prod_D'),
    ('evt_025', 'user_5', 'sess_08', '{day(-2)} 09:30:00', 'purchase',    'prod_A')
''').collect()

print(f'EVENTS: {session.table(EVENTS_TABLE).count()} rows')
session.table(EVENTS_TABLE).show()

EVENTS: 25 rows
---------------------------------------------------------------------------------------------
|"EVENT_ID"  |"USER_ID"  |"SESSION_ID"  |"EVENT_TS"           |"EVENT_TYPE"  |"PRODUCT_ID"  |
---------------------------------------------------------------------------------------------
|evt_001     |user_1     |sess_01       |2026-04-17 15:30:00  |page_view     |prod_A        |
|evt_002     |user_1     |sess_01       |2026-04-17 15:31:00  |page_view     |prod_B        |
|evt_003     |user_1     |sess_01       |2026-04-17 15:35:00  |add_to_cart   |prod_A        |
|evt_004     |user_1     |sess_01       |2026-04-17 15:40:00  |purchase      |prod_A        |
|evt_005     |user_1     |sess_02       |2026-04-19 09:50:00  |page_view     |prod_C        |
|evt_006     |user_1     |sess_02       |2026-04-19 09:55:00  |page_view     |prod_D        |
|evt_007     |user_1     |sess_02       |2026-04-19 10:00:00  |purchase      |prod_C        |
|evt_008     |user_2     |sess_03       |202

In [9]:
# === SESSIONS: Browsing sessions with conversion labels ===
# (Used later as the training spine for conversion prediction)
SESSIONS_TABLE = f'{DATABASE}.{SOURCE_SCHEMA}.SESSIONS'

session.sql(f'''
CREATE OR REPLACE TABLE {SESSIONS_TABLE} (
    SESSION_ID       VARCHAR,
    USER_ID          VARCHAR,
    SESSION_START_TS TIMESTAMP_NTZ,
    SESSION_END_TS   TIMESTAMP_NTZ,
    PAGE_VIEWS       INT,
    IS_CONVERTED     BOOLEAN
)
''').collect()

session.sql(f'''
INSERT INTO {SESSIONS_TABLE} VALUES
    ('sess_01', 'user_1', '{day(-3)} 15:30:00', '{day(-3)} 15:45:00', 3, TRUE),
    ('sess_02', 'user_1', '{day(-1)} 09:50:00', '{day(-1)} 10:05:00', 3, TRUE),
    ('sess_03', 'user_2', '{day(-5)} 11:45:00', '{day(-5)} 12:05:00', 3, TRUE),
    ('sess_04', 'user_2', '{day(-2)} 15:00:00', '{day(-2)} 15:35:00', 3, TRUE),
    ('sess_05', 'user_3', '{day(-6)} 08:30:00', '{day(-6)} 09:05:00', 3, TRUE),
    ('sess_06', 'user_3', '{day(-1)} 13:30:00', '{day(-1)} 14:05:00', 3, TRUE),
    ('sess_07', 'user_4', '{day(-4)} 16:30:00', '{day(-4)} 17:05:00', 3, TRUE),
    ('sess_08', 'user_5', '{day(-2)} 09:00:00', '{day(-2)} 09:35:00', 3, TRUE),
    ('sess_09', 'user_1', '{day(-5)} 20:00:00', '{day(-5)} 20:10:00', 2, FALSE),
    ('sess_10', 'user_3', '{day(-3)} 11:00:00', '{day(-3)} 11:05:00', 1, FALSE),
    ('sess_11', 'user_4', '{day(-8)} 14:00:00', '{day(-8)} 14:15:00', 4, FALSE),
    ('sess_12', 'user_5', '{day(-7)} 18:00:00', '{day(-7)} 18:03:00', 1, FALSE)
''').collect()

print(f'SESSIONS: {session.table(SESSIONS_TABLE).count()} rows')
session.table(SESSIONS_TABLE).show()

SESSIONS: 12 rows
--------------------------------------------------------------------------------------------------------
|"SESSION_ID"  |"USER_ID"  |"SESSION_START_TS"   |"SESSION_END_TS"     |"PAGE_VIEWS"  |"IS_CONVERTED"  |
--------------------------------------------------------------------------------------------------------
|sess_01       |user_1     |2026-04-17 15:30:00  |2026-04-17 15:45:00  |3             |True            |
|sess_02       |user_1     |2026-04-19 09:50:00  |2026-04-19 10:05:00  |3             |True            |
|sess_03       |user_2     |2026-04-15 11:45:00  |2026-04-15 12:05:00  |3             |True            |
|sess_04       |user_2     |2026-04-18 15:00:00  |2026-04-18 15:35:00  |3             |True            |
|sess_05       |user_3     |2026-04-14 08:30:00  |2026-04-14 09:05:00  |3             |True            |
|sess_06       |user_3     |2026-04-19 13:30:00  |2026-04-19 14:05:00  |3             |True            |
|sess_07       |user_4     |2026-04-1

---
## 3. Register Entity

Entities define the join keys used to look up features. Our primary entity is **USER**,
identified by `USER_ID`.

In [10]:
user_entity = Entity(name='USER', join_keys=['USER_ID'], desc='Registered e-commerce user')
fs.register_entity(user_entity)

print('Registered entities:')
fs.list_entities().show()

/Users/dolee/Desktop/snowml-rsureshbabu/snowflake/ml/feature_store/feature_store.py:355: UserWarning: Entity USER already exists. Skip registration.
  return f(self, *args, **kargs)


Registered entities:
--------------------------------------------------------------------
|"NAME"  |"JOIN_KEYS"  |"DESC"                      |"OWNER"       |
--------------------------------------------------------------------
|USER    |["USER_ID"]  |Registered e-commerce user  |ACCOUNTADMIN  |
--------------------------------------------------------------------



---
## 4. Batch Feature View — User Behavior Profile

A **batch feature view** computes pre-aggregated features using SQL.
Here we build a **user behavior profile** by joining data from multiple source tables
(orders, events, sessions) to create a rich feature set for conversion prediction.

Features include:
- **Purchase stats**: order count, total spend, average order value
- **Engagement signals**: event count, distinct products viewed, session count
- **Derived ratios**: conversion rate (orders / sessions), avg events per session

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `timestamp_col` | `LAST_ACTIVITY_TS` | Point-in-time correct joins |
| `refresh_freq` | `1m` | Dynamic Table refresh interval |
| `online_config` | Postgres, 10s lag | Online Feature Table materialization |

In [11]:
# Build a rich user behavior profile by joining orders, events, and sessions
purchase_stats_df = session.sql(f'''
    WITH order_stats AS (
        SELECT
            USER_ID,
            COUNT(ORDER_ID)                AS ORDER_CNT,
            SUM(TOTAL_AMT)                 AS TOTAL_SPEND,
            AVG(TOTAL_AMT)                 AS AVG_ORDER_AMT,
            MAX(TOTAL_AMT)                 AS MAX_ORDER_AMT,
            MAX(ORDER_TS)                  AS LAST_ORDER_TS
        FROM {ORDERS_TABLE}
        GROUP BY USER_ID
    ),
    event_stats AS (
        SELECT
            USER_ID,
            COUNT(EVENT_ID)                AS EVENT_CNT,
            COUNT(DISTINCT PRODUCT_ID)     AS PRODUCTS_VIEWED,
            SUM(CASE WHEN EVENT_TYPE = 'add_to_cart' THEN 1 ELSE 0 END) AS CART_ADDS,
            MAX(EVENT_TS)                  AS LAST_EVENT_TS
        FROM {EVENTS_TABLE}
        GROUP BY USER_ID
    ),
    session_stats AS (
        SELECT
            USER_ID,
            COUNT(SESSION_ID)              AS SESSION_CNT,
            SUM(PAGE_VIEWS)                AS TOTAL_PAGE_VIEWS,
            AVG(PAGE_VIEWS)                AS AVG_PAGE_VIEWS_PER_SESSION
        FROM {SESSIONS_TABLE}
        GROUP BY USER_ID
    )
    SELECT
        o.USER_ID,
        -- Purchase features
        o.ORDER_CNT,
        o.TOTAL_SPEND,
        o.AVG_ORDER_AMT,
        o.MAX_ORDER_AMT,
        -- Engagement features
        e.EVENT_CNT,
        e.PRODUCTS_VIEWED,
        e.CART_ADDS,
        s.SESSION_CNT,
        s.TOTAL_PAGE_VIEWS,
        -- Derived ratios
        ROUND(o.ORDER_CNT / NULLIF(s.SESSION_CNT, 0), 4) AS CONVERSION_RATE,
        ROUND(e.EVENT_CNT / NULLIF(s.SESSION_CNT, 0), 1) AS AVG_EVENTS_PER_SESSION,
        ROUND(e.CART_ADDS / NULLIF(e.EVENT_CNT, 0), 4)   AS CART_ADD_RATE,
        -- Timestamp for PIT joins (latest activity across all sources)
        GREATEST(o.LAST_ORDER_TS, e.LAST_EVENT_TS)        AS LAST_ACTIVITY_TS
    FROM order_stats o
    JOIN event_stats e ON o.USER_ID = e.USER_ID
    JOIN session_stats s ON o.USER_ID = s.USER_ID
''')

purchase_stats_fv = FeatureView(
    name='USER_BEHAVIOR_PROFILE',
    entities=[user_entity],
    feature_df=purchase_stats_df,
    timestamp_col='LAST_ACTIVITY_TS',
    refresh_freq='1m',
    online_config=OnlineConfig(
        enable=True,
        target_lag='10s',
        store_type=OnlineStoreType.POSTGRES,
    ),
    desc='User behavior profile: purchases + engagement + derived ratios from orders, events, sessions',
)

print(f'Defined: {purchase_stats_fv.name} (batch, multi-source join)')
print(f'Features: ORDER_CNT, TOTAL_SPEND, AVG_ORDER_AMT, MAX_ORDER_AMT,')
print(f'          EVENT_CNT, PRODUCTS_VIEWED, CART_ADDS, SESSION_CNT, TOTAL_PAGE_VIEWS,')
print(f'          CONVERSION_RATE, AVG_EVENTS_PER_SESSION, CART_ADD_RATE')

Defined: USER_BEHAVIOR_PROFILE (batch, multi-source join)
Features: ORDER_CNT, TOTAL_SPEND, AVG_ORDER_AMT, MAX_ORDER_AMT,
          EVENT_CNT, PRODUCTS_VIEWED, CART_ADDS, SESSION_CNT, TOTAL_PAGE_VIEWS,
          CONVERSION_RATE, AVG_EVENTS_PER_SESSION, CART_ADD_RATE


In [12]:
# Register and wait for offline backfill
registered_purchase_fv = fs.register_feature_view(purchase_stats_fv, 'V1', overwrite=True)
print(f'Registered: {registered_purchase_fv.name}/{registered_purchase_fv.version}')
print(f'Status: {registered_purchase_fv.status}')

# Wait for Dynamic Table to populate
for _ in range(20):
    count = fs.read_feature_view(registered_purchase_fv, store_type='offline').count()
    print(f'  Offline rows: {count}')
    if count > 0:
        print('Offline backfill complete.')
        break
    time.sleep(15)

print('\nOffline data (user behavior profiles):')
fs.read_feature_view(registered_purchase_fv, store_type='offline').show()


POSTGRES online store type is in private preview since 1.33.0. Do not use in production.
/Users/dolee/Desktop/snowml-rsureshbabu/snowflake/ml/feature_store/feature_store.py:3560: UserWarning: Your pipeline won't be incrementally refreshed due to: "Query contains the aggregation function on a float-typed expression as projection with join in same query block, which is not supported for change tracking. Please consider replacing the floating point expression with a fixed-point number.".
  self._check_dynamic_table_refresh_mode(feature_view_name)


Registered: USER_BEHAVIOR_PROFILE/V1
Status: FeatureViewStatus.ACTIVE
  Offline rows: 5
Offline backfill complete.

Offline data (user behavior profiles):
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"USER_ID"  |"ORDER_CNT"  |"TOTAL_SPEND"  |"AVG_ORDER_AMT"    |"MAX_ORDER_AMT"  |"EVENT_CNT"  |"PRODUCTS_VIEWED"  |"CART_ADDS"  |"SESSION_CNT"  |"TOTAL_PAGE_VIEWS"  |"CONVERSION_RATE"  |"AVG_EVENTS_PER_SESSION"  |"CART_ADD_RATE"  |"LAST_ACTIVITY_TS"   |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|user_2     |4            |990.5          |247.625            |550.0            |6 

---
## 5. Time-Windowed Aggregations

Use the `Feature` class to define rolling-window aggregate features like
*"total spend in the last 7 days"* or *"number of orders in the last 30 days"*.
Under the hood, the Feature Store pre-computes partial aggregates (tiles) and
merges them at query time for efficient, point-in-time correct results.

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `features` | SUM, COUNT, AVG | Aggregation functions over trailing windows |
| `feature_granularity` | `1h` | Granularity of partial aggregation |
| `timestamp_col` | `ORDER_TS` | Event timestamp for windowing |

In [13]:
# Define time-windowed features using the Feature class
purchase_agg_features = [
    Feature.sum('TOTAL_AMT', '7d').alias('SPEND_SUM_7D'),
    Feature.sum('TOTAL_AMT', '30d').alias('SPEND_SUM_30D'),
    Feature.count('ORDER_ID', '7d').alias('ORDER_CNT_7D'),
    Feature.count('ORDER_ID', '30d').alias('ORDER_CNT_30D'),
    Feature.avg('TOTAL_AMT', '7d').alias('AVG_ORDER_AMT_7D'),
]

orders_df = session.table(ORDERS_TABLE)

purchase_agg_fv = FeatureView(
    name='USER_PURCHASE_TRENDS',
    entities=[user_entity],
    feature_df=orders_df,
    timestamp_col='ORDER_TS',
    refresh_freq='1m',
    feature_granularity='1h',
    features=purchase_agg_features,
    online_config=OnlineConfig(
        enable=True,
        target_lag='10s',
        store_type=OnlineStoreType.POSTGRES,
    ),
    desc='User purchase trends: 7d and 30d windows over order amounts',
)

print(f'Defined: {purchase_agg_fv.name} (tiled, {len(purchase_agg_features)} features)')
for f in purchase_agg_features:
    print(f'  - {f}')

Defined: USER_PURCHASE_TRENDS (tiled, 5 features)
  - Feature(sum, column='TOTAL_AMT', window='7d', alias='SPEND_SUM_7D')
  - Feature(sum, column='TOTAL_AMT', window='30d', alias='SPEND_SUM_30D')
  - Feature(count, column='ORDER_ID', window='7d', alias='ORDER_CNT_7D')
  - Feature(count, column='ORDER_ID', window='30d', alias='ORDER_CNT_30D')
  - Feature(avg, column='TOTAL_AMT', window='7d', alias='AVG_ORDER_AMT_7D')


In [ ]:
# Register and wait for offline backfill
registered_agg_fv = fs.register_feature_view(purchase_agg_fv, 'V1', overwrite=False)
print(f'Registered: {registered_agg_fv.name}/{registered_agg_fv.version}')

for _ in range(20):
    count = fs.read_feature_view(registered_agg_fv, store_type='offline').count()
    print(f'  Offline rows: {count}')
    if count > 0:
        print('Offline backfill complete.')
        break
    time.sleep(15)

print('\nOffline data (tiled aggregations):')
fs.read_feature_view(registered_agg_fv, store_type='offline').show()

POSTGRES online store type is in private preview since 1.33.0. Do not use in production.


Registered: USER_PURCHASE_TRENDS/V1
  Offline rows: 5
Offline backfill complete.

Offline data (tiled aggregations):
--------------------------------------------------------------------------------------------------------
|"USER_ID"  |"SPEND_SUM_7D"  |"SPEND_SUM_30D"  |"ORDER_CNT_7D"  |"ORDER_CNT_30D"  |"AVG_ORDER_AMT_7D"  |
--------------------------------------------------------------------------------------------------------
|user_2     |365.5           |990.5            |2               |4                |182.75              |
|user_1     |299.99          |500.49           |2               |5                |149.995             |
|user_4     |95.0            |155.0            |1               |2                |95.0                |
|user_5     |175.0           |575.0            |1               |2                |175.0               |
|user_3     |235.0           |1435.0           |2               |3                |117.5               |
-------------------------------------------

---
## 6. Stream Feature View

A **stream feature view** ingests events via streaming and serves updated features
with **2–3 second end-to-end freshness** — critical for stream personalization
and fraud detection.

Building a stream feature view requires three components:

1. **`StreamSource`** — Defines the event schema. Column names and types must
   exactly match what is sent through the REST ingest API. Register it with
   `fs.register_stream_source()` before use.
2. **`StreamConfig`** — Bundles the stream source, a Python **transformation function** , and historical **backfill data** into a
   single configuration. The transformation function runs server-side on both
   backfill data and incoming events.
3. **`FeatureView`** — Pass `stream_config` instead of `feature_df`. The online
   store is always enabled with `OnlineStoreType.POSTGRES`.

**Flow**: Register StreamSource → Create backfill table → Define StreamConfig →
Register FeatureView → Ingest new events via REST API → Read online

### Step 1a: Register a Stream Source

Define and register a `StreamSource` with the event schema. Column names and types
must exactly match what is sent through the REST ingest API.

In [15]:
# Define the stream source schema
event_stream = StreamSource(
    name='CLICKSTREAM_EVENTS',
    schema=StructType([
        StructField('USER_ID', StringType()),
        StructField('EVENT_TS', TimestampType(TimestampTimeZone.NTZ)),
        StructField('EVENT_TYPE', StringType()),
        StructField('PRODUCT_ID', StringType()),
    ]),
    desc='Stream clickstream events for user activity tracking',
)

fs.register_stream_source(event_stream)
print(f'Stream source registered: {event_stream.name}')

FeatureStore.register_stream_source() is in private preview since 1.33.0. Do not use it in production. 


Stream source registered: CLICKSTREAM_EVENTS


/Users/dolee/anaconda3/lib/python3.11/site-packages/snowflake/snowpark/_internal/utils.py:1172: UserWarning: StreamSource CLICKSTREAM_EVENTS already exists. Skip registration.
  return func(*args, **kwargs)


### Step 1b: Prepare Backfill Data

Select historical events to pre-populate the online store so features are available
immediately — before any new events arrive via the REST ingest API.

In [16]:
backfill_df = session.table(EVENTS_TABLE).select('USER_ID', 'EVENT_TS', 'EVENT_TYPE', 'PRODUCT_ID')
print(f'Backfill: {backfill_df.count()} rows')
backfill_df.show()

Backfill: 25 rows
-----------------------------------------------------------------
|"USER_ID"  |"EVENT_TS"           |"EVENT_TYPE"  |"PRODUCT_ID"  |
-----------------------------------------------------------------
|user_1     |2026-04-17 15:30:00  |page_view     |prod_A        |
|user_1     |2026-04-17 15:31:00  |page_view     |prod_B        |
|user_1     |2026-04-17 15:35:00  |add_to_cart   |prod_A        |
|user_1     |2026-04-17 15:40:00  |purchase      |prod_A        |
|user_1     |2026-04-19 09:50:00  |page_view     |prod_C        |
|user_1     |2026-04-19 09:55:00  |page_view     |prod_D        |
|user_1     |2026-04-19 10:00:00  |purchase      |prod_C        |
|user_2     |2026-04-15 11:45:00  |page_view     |prod_B        |
|user_2     |2026-04-15 11:50:00  |add_to_cart   |prod_B        |
|user_2     |2026-04-15 12:00:00  |purchase      |prod_B        |
-----------------------------------------------------------------



### Step 2: Define the Stream Feature View

Create a `FeatureView` with a `StreamConfig` that bundles:
- **`stream_source`** — the registered `StreamSource` schema
- **`transformation_fn`** — a simple Python function that processes events using pandas, applied server-side to both backfill and incoming events. Here `process_events` filters to known event types, drops rows with missing `USER_ID` or `PRODUCT_ID`, and derives an `IS_PURCHASE` flag
- **`backfill_df`** — historical data to pre-populate the online store

In [17]:
import pandas as pd

# Transformation function for the stream feature view.
# Takes a pandas DataFrame of raw events and returns a cleaned/enriched DataFrame.
def process_events(df: pd.DataFrame) -> pd.DataFrame:
    # Filter to only known event types (drop unexpected/malformed events)
    df = df[df['EVENT_TYPE'].isin(['page_view', 'add_to_cart', 'purchase'])]
    # Drop rows missing required fields
    df = df[df['USER_ID'].notna() & (df['USER_ID'] != '')]
    df = df[df['PRODUCT_ID'].notna() & (df['PRODUCT_ID'] != '')]
    # Derive a binary purchase flag for downstream models
    df['IS_PURCHASE'] = (df['EVENT_TYPE'] == 'purchase').astype(int)
    return df

In [18]:
stream_fv = FeatureView(
    name='USER_STREAM_EVENTS',
    entities=[user_entity],
    timestamp_col='EVENT_TS',
    stream_config=StreamConfig(
        stream_source=event_stream,
        transformation_fn=process_events,
        backfill_df=backfill_df,
    ),
    online_config=OnlineConfig(
        enable=True,
        target_lag='10s',
        store_type=OnlineStoreType.POSTGRES,
    ),
    desc='Processed clickstream events: filtered and enriched for online serving',
)

### Step 3: Register the Stream Feature View

Register the feature view with `overwrite=True`. This creates the backing infrastructure
(Dynamic Table, online table) and begins backfilling historical data into the online store.

In [19]:
# Register
registered_stream_fv = fs.register_feature_view(stream_fv, 'V1', overwrite=True)
print(f'Registered: {registered_stream_fv.name}/{registered_stream_fv.version}')
print(f'Status: {registered_stream_fv.status}')

POSTGRES online store type is in private preview since 1.33.0. Do not use in production.


Registered: USER_STREAM_EVENTS/V1
Status: FeatureViewStatus.STATIC


*For demo purposes, let's save the current online features before and after ingestion so we can compare them side by side.*

In [20]:
import time

stream_fv_obj = fs.get_feature_view('USER_STREAM_EVENTS', 'V1')
keys = [['user_1'], ['user_2']]

print('=== BEFORE Ingest: Online Features for USER_STREAM_EVENTS ===')
before_df = fs.read_feature_view(stream_fv_obj, keys=keys, store_type='online')
before_pd = before_df.to_pandas()
print(before_pd.to_string(index=False))

=== BEFORE Ingest: Online Features for USER_STREAM_EVENTS ===
USER_ID  EVENT_TYPE PRODUCT_ID  IS_PURCHASE
 user_1   page_view     prod_D            0
 user_2 add_to_cart     prod_B            0


### Step 4: Ingest Events

Use `fs.stream_ingest()` to send new events into a stream feature view. Events are
pushed over the REST ingest API and become available in the online store within seconds.

```python
fs.stream_ingest(
    stream_source='CLICKSTREAM_EVENTS',
    records=[
        {'USER_ID': 'user_1', 'EVENT_TS': '2026-04-17 10:00:00', 'EVENT_TYPE': 'page_view', 'PRODUCT_ID': 'prod_F'},
    ],
)
```

In [21]:
new_events = [
    {'USER_ID': 'user_1', 'EVENT_TS': f'{day(0)} 10:00:00', 'EVENT_TYPE': 'page_view',   'PRODUCT_ID': 'prod_F'},
    {'USER_ID': 'user_1', 'EVENT_TS': f'{day(0)} 10:05:00', 'EVENT_TYPE': 'add_to_cart', 'PRODUCT_ID': 'prod_F'},
    {'USER_ID': 'user_2', 'EVENT_TS': f'{day(0)} 10:10:00', 'EVENT_TYPE': 'page_view',   'PRODUCT_ID': 'prod_A'},
]

print(f'=== Ingesting {len(new_events)} new events ===')
for e in new_events:
    print(f"  → {e['USER_ID']}  {e['EVENT_TS']}  {e['EVENT_TYPE']:14s}  {e['PRODUCT_ID']}")

accepted = fs.stream_ingest(
    stream_source='CLICKSTREAM_EVENTS',
    records=new_events,
)
print(f'\n✓ Ingested {accepted} events')

=== Ingesting 3 new events ===
  → user_1  2026-04-20 10:00:00  page_view       prod_F
  → user_1  2026-04-20 10:05:00  add_to_cart     prod_F
  → user_2  2026-04-20 10:10:00  page_view       prod_A

✓ Ingested 3 events


Alternatively, you can POST events directly to the Ingest REST API endpoint. The endpoint URLs were printed when the Online Service started (cell above), or you can retrieve them anytime:

In [30]:
status = fs.get_online_service_status()
ingest_url = online_service.endpoint_url(status, 'ingest')
print(ingest_url)

https://bxhi5jn-sfengineering-feature-store-vnext.awsuswest2qa6.test-snowflakecomputing.app


Put your PAT and ingest URL in the curl command:

```bash
curl -X POST \
  "{ingest_url}/api/v1/ingest" \
  -H 'Authorization: Snowflake Token="<your_PAT>"' \
  -H 'Content-Type: application/json' \
  -d '{{
    "records": {{
      "CLICKSTREAM_EVENTS": [
        {{"USER_ID": "user_1", "EVENT_TS": "{day(0)} 10:00:00", "EVENT_TYPE": "page_view", "PRODUCT_ID": "prod_F"}}
      ]
    }}
  }}'
```

Poll the online store every second until the newly ingested events appear. This
measures the **end-to-end freshness** — the time from ingest to when updated features
are queryable in the online store.

In [22]:
print('=== Waiting for freshness (polling online store) ===')
poll_start = time.time()
freshness_s = None
for attempt in range(30):
    time.sleep(1)
    after_df = fs.read_feature_view(stream_fv_obj, keys=keys, store_type='online')
    after_pd = after_df.to_pandas()
    if not after_pd.equals(before_pd):
        freshness_s = time.time() - poll_start
        break

if freshness_s is not None:
    print(f'\n✓ End-to-end freshness: {freshness_s:.1f}s (ingest → online store)')
else:
    after_pd = before_pd
    print(f'⚠ Data unchanged after 30s — stream may still be processing')

=== Waiting for freshness (polling online store) ===

✓ End-to-end freshness: 1.5s (ingest → online store)


### Compare Before vs After

The online store is **keyed by entity** (`USER_ID`), so it maintains the **latest state** per user. 
Ingesting new events for `user_1` and `user_2` **updates** their existing rows, `user_3` is new. The comparison below highlights which rows changed.

In [23]:
key_col = 'USER_ID'
cols = [c for c in after_pd.columns if c in before_pd.columns]

print('BEFORE:')
print(before_pd[cols].to_string(index=False))

print('\nSTREAM EVENTS INGESTED:')
for e in new_events:
    print(f"{e['EVENT_TS']}  {e['USER_ID']}  {e['EVENT_TYPE']:14s}  {e['PRODUCT_ID']}")

print('\nAFTER:')
print(after_pd[cols].to_string(index=False))

print('\nChanges:')
for uid in before_pd[key_col]:
    b = before_pd[before_pd[key_col] == uid].iloc[0]
    a = after_pd[after_pd[key_col] == uid].iloc[0]
    diffs = [c for c in cols if c != key_col and str(b[c]) != str(a[c])]
    if diffs:
        for c in diffs:
            print(f'  {uid}.{c}: {b[c]} → {a[c]}')

if freshness_s is not None:
    print(f'\n✓ End-to-end freshness: {freshness_s:.1f}s')

BEFORE:
USER_ID  EVENT_TYPE PRODUCT_ID  IS_PURCHASE
 user_1   page_view     prod_D            0
 user_2 add_to_cart     prod_B            0

STREAM EVENTS INGESTED:
2026-04-20 10:00:00  user_1  page_view       prod_F
2026-04-20 10:05:00  user_1  add_to_cart     prod_F
2026-04-20 10:10:00  user_2  page_view       prod_A

AFTER:
USER_ID  EVENT_TYPE PRODUCT_ID  IS_PURCHASE
 user_1 add_to_cart     prod_F            0
 user_2   page_view     prod_A            0

Changes:
  user_1.EVENT_TYPE: page_view → add_to_cart
  user_1.PRODUCT_ID: prod_D → prod_F
  user_2.EVENT_TYPE: add_to_cart → page_view
  user_2.PRODUCT_ID: prod_B → prod_A

✓ End-to-end freshness: 1.5s


---
## 7. Online Retrieval

Read features from the **Postgres online store** for low-latency serving.
The SDK's `read_feature_view(..., store_type="online")` wraps the REST call in a Snowpark DataFrame,
which adds overhead. Below we also show a **direct HTTP request** to the Query API so you can see
the raw Postgres REST latency (~20 ms).

> **Note:** Online reads require a valid `SNOWFLAKE_PAT` environment variable and
> your IP address must be allowed by the online service's network policy.

In [ ]:
keys = [['user_1'], ['user_2'], ['user_3']]

fv = fs.get_feature_view('USER_BEHAVIOR_PROFILE', 'V1')
print(f'=== Online: {fv.name}/{fv.version} ===')
result = fs.read_feature_view(fv, keys=keys, store_type='online')
result.show()

=== Online: USER_BEHAVIOR_PROFILE/V1 ===
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"USER_ID"  |"ORDER_CNT"  |"TOTAL_SPEND"  |"AVG_ORDER_AMT"    |"MAX_ORDER_AMT"  |"EVENT_CNT"  |"PRODUCTS_VIEWED"  |"CART_ADDS"  |"SESSION_CNT"  |"TOTAL_PAGE_VIEWS"  |"CONVERSION_RATE"  |"AVG_EVENTS_PER_SESSION"  |"CART_ADD_RATE"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|user_1     |5            |500.49         |100.098            |210.0            |7            |4                  |1            |3              |8                   |2                  |2                         |0                |
|user_2     |4            |990.

Equivalent curl command for the query API:

```bash
curl -X POST \
  "<query_url>/api/v1/query" \
  -H 'Authorization: Snowflake Token="<your_PAT>"' \
  -H 'Content-Type: application/json' \
  -d '{
    "feature_view": "USER_PURCHASE_TRENDS",
    "version": "V1",
    "keys": [
      {"USER_ID": "user_1"},
      {"USER_ID": "user_2"}
    ]
  }'
```

---
## 8. Generate Training Set

Create a **point-in-time correct training set** for session conversion prediction.

The spine is the SESSIONS table — each row represents a browsing session with:
- `USER_ID` — the entity key
- `SESSION_START_TS` — the point-in-time timestamp
- `IS_CONVERTED` — the label (did the user purchase?)

The Feature Store joins features from multiple feature views **as of** each session's
start time, ensuring no data leakage from the future.

In [37]:
# Build the training spine from SESSIONS
spine_df = session.sql(f'''
    SELECT
        USER_ID,
        SESSION_START_TS,
        IS_CONVERTED AS LABEL
    FROM {SESSIONS_TABLE}
''')

print('Training spine:')
spine_df.show()

Training spine:
---------------------------------------------
|"USER_ID"  |"SESSION_START_TS"   |"LABEL"  |
---------------------------------------------
|user_1     |2026-04-17 15:30:00  |True     |
|user_1     |2026-04-19 09:50:00  |True     |
|user_2     |2026-04-15 11:45:00  |True     |
|user_2     |2026-04-18 15:00:00  |True     |
|user_3     |2026-04-14 08:30:00  |True     |
|user_3     |2026-04-19 13:30:00  |True     |
|user_4     |2026-04-16 16:30:00  |True     |
|user_5     |2026-04-18 09:00:00  |True     |
|user_1     |2026-04-15 20:00:00  |False    |
|user_3     |2026-04-17 11:00:00  |False    |
---------------------------------------------



In [38]:
# Generate training set with point-in-time correct joins
training_df = fs.generate_training_set(
    spine_df=spine_df,
    features=[registered_agg_fv],
    spine_timestamp_col='SESSION_START_TS',
    spine_label_cols=['LABEL'],
    join_method="cte"
)

print(f'Training set: {training_df.count()} rows')
print(f'Columns: {training_df.columns}')
training_df.show()

Training set: 12 rows
Columns: ['USER_ID', 'SESSION_START_TS', 'LABEL', 'SPEND_SUM_7D', 'SPEND_SUM_30D', 'ORDER_CNT_7D', 'ORDER_CNT_30D', 'AVG_ORDER_AMT_7D']
----------------------------------------------------------------------------------------------------------------------------------------
|"USER_ID"  |"SESSION_START_TS"   |"LABEL"  |"SPEND_SUM_7D"  |"SPEND_SUM_30D"  |"ORDER_CNT_7D"  |"ORDER_CNT_30D"  |"AVG_ORDER_AMT_7D"  |
----------------------------------------------------------------------------------------------------------------------------------------
|user_3     |2026-04-14 08:30:00  |True     |0.0             |1200.0           |0               |1                |NULL                |
|user_3     |2026-04-19 13:30:00  |True     |85.0            |1285.0           |1               |2                |85.0                |
|user_4     |2026-04-16 16:30:00  |True     |60.0            |60.0             |1               |1                |60.0                |
|user_3     |2026-04

---
## 9. Inspect Feature Store

Review all registered feature views and their configurations.

In [39]:
print('=== Entities ===')
fs.list_entities().show()

print('\n=== Feature Views ===')
fs.list_feature_views().select(
    'NAME', 'VERSION', 'SCHEDULING_STATE', 'ONLINE_CONFIG'
).show()

print('\n=== Online Service ===')
try:
    status = fs.get_online_service_status()
    print(f'Status: {status.status}')
    print(f'Endpoints: {[ep.name for ep in status.endpoints]}')
except Exception as e:
    print(f'Status check failed: {e}')

=== Entities ===
--------------------------------------------------------------------
|"NAME"  |"JOIN_KEYS"  |"DESC"                      |"OWNER"       |
--------------------------------------------------------------------
|USER    |["USER_ID"]  |Registered e-commerce user  |ACCOUNTADMIN  |
--------------------------------------------------------------------


=== Feature Views ===
------------------------------------------------------------------------------------------------------------------
|"NAME"                    |"VERSION"  |"SCHEDULING_STATE"  |"ONLINE_CONFIG"                                     |
------------------------------------------------------------------------------------------------------------------
|USER_BEHAVIOR_PROFILE     |V1         |ACTIVE              |{"enable": true, "target_lag": "10 seconds", "s...  |
|USER_PURCHASE_AGGREGATES  |V1         |ACTIVE              |{"enable": true, "target_lag": "10 seconds", "s...  |
|USER_PURCHASE_STATS       |V1         

---
## 10. Cleanup (Optional)

Uncomment the cells below to clean up all resources created by this quickstart.

In [40]:
# # Uncomment to clean up:
# fs.drop_online_service()
# fs.delete_feature_view(fs.get_feature_view('USER_STREAM_EVENTS', 'V1'))
# fs.delete_feature_view(fs.get_feature_view('USER_PURCHASE_TRENDS', 'V1'))
# fs.delete_feature_view(fs.get_feature_view('USER_BEHAVIOR_PROFILE', 'V1'))
# fs.delete_entity('USER')
# session.sql(f'DROP DATABASE IF EXISTS {DATABASE}').collect()
# print('Cleanup complete.')

print('Cleanup skipped — resources preserved.')

Cleanup skipped — resources preserved.
